In [ ]:
#behaviour based similarity
#downloading the dataset
#my recommendation system suggests movies that were rated similarly by the same group of users.


In [ ]:
import numpy as np
import pandas as pd
import warnings

In [ ]:
warnings.filterwarnings('ignore')

get the dataset

In [ ]:
column_names =  ['user_id','item_id','rating','timestamp']
df = pd.read_csv('/home/megha/projects/movie_recommendation_system/ml-100k/u.data',sep='\t',names = column_names)  #tsv(tab seperated values)

In [ ]:
print(df.head())
print(df.shape)
print(df['user_id'].nunique())
print(df['item_id'].nunique())

In [ ]:
movie_names = pd.read_csv('ml-100k/u.item',sep='|',encoding='ISO-8859-1',header = None)
movie_names = movie_names[[0,1]]
movie_names.columns = ['item_id','title']
print(movie_names.head())

In [ ]:
#now merge both df and movie_names
df = pd.merge(df,movie_names,on='item_id')  #final dataset

Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('white')

In [ ]:
df.groupby('title').mean()['rating'].sort_values(ascending=False).head()

In [ ]:
# Combined statistics: average rating and number of viewers
movie_stats = df.groupby('title').agg({
    'rating': ['count', 'mean']
}).round(2)
movie_stats.columns = ['num_viewers', 'avg_rating']
movie_stats = movie_stats.sort_values('avg_rating', ascending=False)
print(movie_stats)

Histogram of viewers

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(movie_stats['num_viewers'], bins=50, color='skyblue')
plt.title('Distribution of Number of Viewers')
plt.xlabel('Number of Viewers')
plt.ylabel('Frequency(no.of movies)')
plt.show()

Histogram of avg ratings

In [ ]:
plt.hist(movie_stats['avg_rating'], bins=50, color='skyblue')
plt.title('Distribution of Average Ratings')
plt.xlabel('Average Rating')
plt.ylabel('Frequency(no.of movies)')
plt.show()  #most of the users rated movies as 3

number of viewers and ratings

In [ ]:
sns.jointplot(x='avg_rating' ,y='num_viewers' ,data=movie_stats ,alpha=0.5)  #as the rating increases number of viewers also increases

creating movie recommendation

In [ ]:
df.head()

In [ ]:
moviemat = df.pivot_table(index='user_id',columns='title',values='rating')
moviemat.head()


In [ ]:
movie_stats.sort_values('num_viewers',ascending=False).head(10)

In [ ]:
starwars_user_ratings = moviemat['Star Wars (1977)']
starwars_user_ratings.head()

In [ ]:
#now do correlate of one movie with whole movie matrix
similar_to_starwars = moviemat.corrwith(starwars_user_ratings)
corr_starwars = pd.DataFrame(similar_to_starwars,columns=['correlation'])
corr_starwars.dropna(inplace = True)


In [ ]:
corr_starwars.sort_values('correlation',ascending=False)

In [ ]:
corr_starwars = corr_starwars.join(movie_stats['num_viewers'])
corr_starwars[corr_starwars['num_viewers']>100].sort_values('correlation',ascending=False).head(10)

Prediction Function

In [ ]:
def predict_movies(movie_name):
  movie_user_ratings = moviemat[movie_name]
  similar_to_movie = moviemat.corrwith(movie_user_ratings)
  corr_movie = pd.DataFrame(similar_to_movie,columns=['correlation'])
  corr_movie.dropna(inplace = True)
  corr_movie = corr_movie.join(movie_stats['num_viewers'])
  predictions = corr_movie[corr_movie['num_viewers']>100].sort_values(['correlation','num_viewers'],ascending=[False,False]).head(5)
  return predictions

In [ ]:
def show_movie_list():
    """Displays list of available movies for user selection"""
    # Get movies with sufficient ratings (>100 viewers)
    available_movies = movie_stats[movie_stats['num_viewers'] > 100].index.tolist()
    available_movies = sorted(available_movies)

    print(f"📽️  Available Movies ({len(available_movies)} total)")
    print("=" * 70)

    # Display all available movies
    for i, movie in enumerate(available_movies, 1):
        print(f"{i:4d}. {movie}")
        if i % 50 == 0:  # Pause every 50 movies for readability
            input("\nPress Enter to see more movies...")
            print()

    print("=" * 70)



In [ ]:
def get_recommendations():
    """Get recommendations for a movie"""
    # Get user choice
    movie_name = input("\n🎬 Enter the movie name: ")

    # Get movies with sufficient ratings
    available_movies = movie_stats[movie_stats['num_viewers'] > 100].index.tolist()

    # Check if movie exists
    if movie_name not in available_movies:
        print(f"\n❌ Movie not found! Please use exact name from the list.")
        return

    print(f"\nFinding recommendations for: {movie_name}")
    print("-" * 70)

    # Get recommendations
    recommendations = predict_movies(movie_name)
    print(recommendations)

In [ ]:
show_movie_list()

In [ ]:

get_recommendations()

In [ ]:
predict_movies("American Dream (1990)")